# 01 — Exploração: Editais de Leilão DETRAN/MG

**Fase 2 do projeto** — validar o scraping antes de persistir em banco.

**Objetivos:**
1. Entender a requisição HTTP ao portal
2. Inspecionar a estrutura HTML dos cards de edital
3. Construir e validar o parser com `pandas`
4. Conferir qualidade dos dados (nulls, duplicatas, datas)
5. Usar o módulo `detran_scraper` (código reutilizável da Fase 1)

**Fonte:** https://leilao.detran.mg.gov.br/

## 1. Setup e requisição HTTP

O portal usa **CakePHP** com HTML server-side. Precisamos de `User-Agent` de browser e aceitar cookies (`CAKEPHP`) na primeira visita.

In [1]:
import os
import sys
from pathlib import Path

import httpx
import pandas as pd

# raiz do repo (notebooks/ → ..)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

BASE_URL = os.getenv("DETRAN_BASE_URL", "https://leilao.detran.mg.gov.br")
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "pt-BR,pt;q=0.9",
}

with httpx.Client(base_url=BASE_URL, headers=HEADERS, follow_redirects=True, timeout=30) as client:
    response = client.get("/")
    response.raise_for_status()
    html = response.text
    cookies = dict(client.cookies)

print(f"Status: {response.status_code}")
print(f"Tamanho HTML: {len(html):,} bytes")
print(f"Cookies: {list(cookies.keys())}")
print(f"Content-Type: {response.headers.get('content-type')}")

Status: 200
Tamanho HTML: 81,362 bytes
Cookies: ['CAKEPHP', 'ROUTEID']
Content-Type: text/html; charset=UTF-8


## 2. Inspeção do HTML

Localizamos os cards pelo título `h5.capa-titulo`. Cada card contém município, pátio, status, data de encerramento e link para lotes.

In [2]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "lxml")
cards = soup.select("h5.capa-titulo")
print(f"Cards encontrados (h5.capa-titulo): {len(cards)}")

# amostra do primeiro card
first_card = cards[0].find_parent("div", class_="card")
print(first_card.prettify()[:1200])

Cards encontrados (h5.capa-titulo): 21
<div class="card">
 <img alt="" class="card-img-top" src="/../../webroot/img/capadoleilao.png"/>
 <div class="card-body text-center col-12 capa-card">
  <h5 class="card-title capa-titulo">
   <b>
    Edital de Leilão
    <br/>
    1692/2026
   </b>
  </h5>
  <p class="card-text capa-municipio">
   joao monlevade
  </p>
 </div>
 <div class="card-body p-1 border-top">
  <div class="row">
   <div class="col-12 text-center align-self-center" style="height: 45px; font-size:12px">
    <b>
     1692 - JS SERVICOS DE REBOQUE E ESTACIONAMENTO LTDA
    </b>
   </div>
   <div class="col-12 text-center text-primary">
    Publicado
   </div>
   <div class="col-12 text-center">
    Encerramento: 21/08/2026 17:55
   </div>
  </div>
 </div>
 <div class="card-footer p-2">
  <a class="btn btn-primary btn-block" href="/lotes/lista-lotes/3416/2026">
   <i aria-hidden="true" class="fa fa-fas fa-gavel">
   </i>
   Detalhes
  </a>
 </div>
</div>



## 3. Parser v0 (inline)

Construímos o parser diretamente no notebook para entender cada campo antes de extrair para o módulo.

In [3]:
import re
from datetime import datetime
from urllib.parse import urljoin

NUMERO_RE = re.compile(r"\d+/\d+")
PATIO_RE = re.compile(r"^\s*\d+\s*-\s*(.+)$")
ENCERRAMENTO_RE = re.compile(r"Encerramento:\s*(\d{2}/\d{2}/\d{4})\s+(\d{2}:\d{2})")
LOTE_URL_RE = re.compile(r"/lotes/lista-lotes/(\d+)/(\d+)")


def parse_card_v0(card) -> dict | None:
    title = card.select_one("h5.capa-titulo")
    numero_match = NUMERO_RE.search(title.get_text()) if title else None
    municipio_el = card.select_one("p.capa-municipio")
    patio_el = card.select_one("div.card-body.p-1.border-top b")
    status_el = card.select_one("div.text-primary, div.text-danger")
    link_el = card.select_one("a[href*='/lotes/lista-lotes/']")

    encerramento = None
    for el in card.select("div.col-12.text-center"):
        m = ENCERRAMENTO_RE.match(el.get_text(strip=True))
        if m:
            encerramento = datetime.strptime(f"{m.group(1)} {m.group(2)}", "%d/%m/%Y %H:%M")
            break

    patio_match = PATIO_RE.match(patio_el.get_text()) if patio_el else None
    lote_match = LOTE_URL_RE.search(link_el["href"]) if link_el and link_el.get("href") else None

    if not all([numero_match, municipio_el, patio_match, status_el, encerramento, lote_match]):
        return None

    return {
        "leilao_id": int(lote_match.group(1)),
        "numero_edital": numero_match.group(),
        "municipio": " ".join(municipio_el.get_text().split()).strip().title(),
        "patio": " ".join(patio_match.group(1).split()).strip().title(),
        "status": status_el.get_text(strip=True).title(),
        "data_encerramento": encerramento,
        "url_detalhes": urljoin(BASE_URL, link_el["href"]),
    }


rows = []
for title in soup.select("h5.capa-titulo"):
    card = title.find_parent("div", class_="card")
    if card:
        row = parse_card_v0(card)
        if row:
            rows.append(row)

df = pd.DataFrame(rows)
df

,leilao_id,numero_edital,municipio,patio,status,data_encerramento,url_detalhes
0,3416,1692/2026,Joao Monlevade,Js Servicos De Reboque E Estacionamento Ltda,Publicado,2026-08-21 17:55:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...
1,3438,1714/2026,Manhuacu,Patio Minas Ltda,Publicado,2026-07-09 17:00:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...
2,3332,1608/2026,Bom Despacho,Socorro Rebocar Ltda,Publicado,2026-07-02 17:55:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...
3,3414,1690/2026,Vicosa,Auto Socorro Joao Rossi Ltda,Publicado,2026-07-03 17:00:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...
4,3415,1691/2026,Vicosa,Joao Donizette Rossi,Publicado,2026-07-10 17:00:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...
5,3435,1711/2026,Arcos,Auto Socorro Lira - Servicos De Guincho Ltda,Publicado,2026-06-30 17:30:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...
6,3399,1675/2026,Nova Serrana,Socorro Serranense Servicos De Reboque Ltda,Finalizado,2026-06-22 17:55:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...
7,3341,1617/2026,Uberlandia,Gran Parking De Apreensoes De Uberlandia Ltda,Publicado,2026-06-30 17:55:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...
8,3424,1700/2026,Itamogi,Dielson Alves Rabelo 81199295604,Publicado,2026-07-09 17:30:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...
9,3336,1612/2026,Muzambinho,J S N Patio Novo Rumo Ltda,Publicado,2026-07-01 17:00:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...


## 4. Qualidade dos dados

Verificamos completude, duplicatas e distribuição de status.

In [4]:
print("Registros:", len(df))
print("\nNulls por coluna:")
print(df.isna().sum())

dup_ids = df[df.duplicated("leilao_id", keep=False)]
print(f"\nDuplicatas por leilao_id: {len(dup_ids)}")

print("\nStatus:")
print(df["status"].value_counts())

print("\nEncerramentos (min / max):")
print(df["data_encerramento"].min(), "→", df["data_encerramento"].max())

assert len(df) >= 15, f"Esperado >= 15 editais, obtido {len(df)}"
assert df["leilao_id"].is_unique, "leilao_id deve ser único"
print("\n✓ Validação básica OK")

Registros: 21

Nulls por coluna:
leilao_id            0
numero_edital        0
municipio            0
patio                0
status               0
data_encerramento    0
url_detalhes         0
dtype: int64

Duplicatas por leilao_id: 0

Status:
status
Publicado     12
Finalizado     9
Name: count, dtype: int64

Encerramentos (min / max):
2026-06-22 17:00:00 → 2026-08-21 17:55:00

✓ Validação básica OK


## 5. Edge cases

- **Status:** `Publicado` usa classe `text-primary`; `Finalizado` usa `text-danger`
- **Pátio:** texto no formato `NUMERO - RAZÃO SOCIAL`, com espaços extras ocasionais

In [5]:
# amostra por status
for status in df["status"].unique():
    sample = df[df["status"] == status].iloc[0]
    print(f"[{status}] {sample['numero_edital']} — {sample['patio'][:50]}...")

# editais que encerram nos próximos 30 dias
from datetime import timedelta

hoje = pd.Timestamp.now().normalize()
proximos = df[
    (df["status"] == "Publicado")
    & (df["data_encerramento"] >= hoje)
    & (df["data_encerramento"] <= hoje + timedelta(days=30))
].sort_values("data_encerramento")

print(f"\nPublicados encerrando em 30 dias: {len(proximos)}")
proximos[["numero_edital", "municipio", "data_encerramento"]]

[Publicado] 1692/2026 — Js Servicos De Reboque E Estacionamento Ltda...
[Finalizado] 1675/2026 — Socorro Serranense Servicos De Reboque Ltda...

Publicados encerrando em 30 dias: 10


,numero_edital,municipio,data_encerramento
5,1711/2026,Arcos,2026-06-30 17:30:00
7,1617/2026,Uberlandia,2026-06-30 17:55:00
9,1612/2026,Muzambinho,2026-07-01 17:00:00
2,1608/2026,Bom Despacho,2026-07-02 17:55:00
3,1690/2026,Vicosa,2026-07-03 17:00:00
15,1620/2026,Governador Valadares,2026-07-03 17:00:00
19,1417/2026,Joao Monlevade,2026-07-03 17:55:00
1,1714/2026,Manhuacu,2026-07-09 17:00:00
8,1700/2026,Itamogi,2026-07-09 17:30:00
4,1691/2026,Vicosa,2026-07-10 17:00:00


## 6. Módulo `detran_scraper` (Fase 1)

A lógica validada acima vive em `src/detran_scraper/`. O notebook reutiliza o mesmo código que o pipeline usará depois.

In [6]:
from dataclasses import asdict

from detran_scraper import DetranClient, parse_editais

with DetranClient(base_url=BASE_URL) as client:
    html_modulo = client.fetch_home()

editais = parse_editais(html_modulo, base_url=BASE_URL)
df_modulo = pd.DataFrame(asdict(e) for e in editais)

# comparar parser inline vs módulo
cols = ["leilao_id", "numero_edital", "municipio", "status"]
pd.testing.assert_frame_equal(
    df[cols].sort_values("leilao_id").reset_index(drop=True),
    df_modulo[cols].sort_values("leilao_id").reset_index(drop=True),
)
print(f"Módulo retornou {len(editais)} editais — idêntico ao parser inline ✓")
df_modulo.head()

Módulo retornou 21 editais — idêntico ao parser inline ✓


,leilao_id,numero_edital,municipio,patio,status,data_encerramento,url_detalhes,scraped_at,raw_hash
0,3416,1692/2026,Joao Monlevade,Js Servicos De Reboque E Estacionamento Ltda,Publicado,2026-08-21 17:55:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...,None,5d2769869ca19b18b3c5492e3a31f1dfbf0c94c5063208...
1,3438,1714/2026,Manhuacu,Patio Minas Ltda,Publicado,2026-07-09 17:00:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...,None,85ed1155501a1e78a0fb8c2af207553b02e6028bad6d43...
2,3332,1608/2026,Bom Despacho,Socorro Rebocar Ltda,Publicado,2026-07-02 17:55:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...,None,68b26469a5b78e3414d0b57420a4f5b116945e524127ac...
3,3414,1690/2026,Vicosa,Auto Socorro Joao Rossi Ltda,Publicado,2026-07-03 17:00:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...,None,d2b27af33a4775e871d69ef3afde6827d6022b5fa3dc3c...
4,3415,1691/2026,Vicosa,Joao Donizette Rossi,Publicado,2026-07-10 17:00:00,https://leilao.detran.mg.gov.br/lotes/lista-lo...,None,cddad8f5a13562778183c65b1d30c2e85cbe75e237c97b...


## Próximos passos (Fase 3+)

- Subir Postgres com `docker compose`
- Persistir `df_modulo` com upsert idempotente
- Gráficos: editais por município, timeline de encerramentos